In [ ]:
from itertools import groupby
import matplotlib.pyplot as plt
import mplhep
import ROOT
import os
import re
import seaborn as sns

sns.set_style("white")
sns.set_context("notebook")
sns.set_palette("colorblind")

ROOT.TH1.AddDirectory(False)

# -----------------------------------------------------------------------------
# Notebook configuration
# -----------------------------------------------------------------------------

# Shapes tag
SHAPES_TAG = "shapes-2026-05-07"

# Era name
ERA = "2024"

# Category name
CATEGORY = "mt_base_sr"

# Base path to the validation database
SHAPES_FILE = os.path.join("/work/mmolch/xyh-bbtautau-crown/bbtautau/data/output/Shapes", SHAPES_TAG, ERA, CATEGORY, "shapes.root")

In [ ]:
def load_histograms(histogram_file: str):
    """
    Load histograms from a ROOT file and parse their key strings.
    
    Key format: dataset#channel-category-process-dataset#shift#variable
    """
    histograms = []
    rf = ROOT.TFile.Open(histogram_file, "READ")
    
    if not rf or rf.IsZombie():
        raise Exception(f"Could not open file: {histogram_file}")
    
    for key in rf.GetListOfKeys():
        key_str = key.GetTitle()
        # Parse histogram key: dataset#channel-category-process-dataset#shift#variable
        match = re.match(r"^([^#]*)#([^#]*)-([^#]*)-([^#]*)-([^#]*)#([^#]*)#([^#]*)", key_str)
        if not match:
            print(f"Warning: Could not parse key: {key_str}")
            continue
        
        histogram_dict = {
            "key": key_str,
            "channel": match.group(2),
            "category": match.group(3),
            "process": match.group(4),
            "dataset": match.group(5),
            "shift": match.group(6),
            "variable": match.group(7),
            "histogram": rf.Get(key_str),
        }
        histograms.append(histogram_dict)
    
    return histograms, rf

In [ ]:
# Load histograms
histograms, root_file = load_histograms(SHAPES_FILE)

print(f"Loaded {len(histograms)} histograms")
print("\nFirst 5 histograms:")
for i, h in enumerate(histograms[:5]):
    print(f"  {i}: {h['key']}")
    print(f"     Channel: {h['channel']}, Category: {h['category']}, Process: {h['process']}, Variable: {h['variable']}")

In [ ]:
# Select a variable to inspect
CHANNEL = "mt"
CATEGORY = "mt_base_sr"
VARIABLE = "m_vis"
SHIFT = "Nominal"

# Filter histograms for the selected parameters and group by process
selected_histograms = (
    h for h in histograms
    if (
        h['channel'] == CHANNEL
        and h['category'] == CATEGORY
        and h['variable'] == VARIABLE
        and h['shift'] == SHIFT
    )
)
grouped_histograms = groupby(sorted(selected_histograms, key=lambda h: h['process']), key=lambda h: h['process'])

print(f"Histograms for variable '{VARIABLE}' in channel '{CHANNEL}' and category '{CATEGORY}'")

for process, group in grouped_histograms:
    # Output message for process
    print("\nProcess:", process)

    # Create figure for this process
    fig, ax = plt.subplots()

    datasets = []
    for h in group:
        # Add dataset information for logging output
        datasets.append(h['dataset'])

        # Get the ROOT histogram
        root_hist = h['histogram']

        # Plot histogram
        mplhep.histplot(root_hist, yerr=True, ax=ax, label=h['dataset'])

    # Style the plot (axes labels, legend, title)
    ax.set_xlabel(VARIABLE)
    ax.set_ylabel('Number of events')
    ax.set_title(f"{process}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Thighten layout
    fig.tight_layout()

    # Output message for datasets
    print(f"Datasets: {', '.join(datasets)}")

    # Show the plot and clear the figure for the next process
    plt.show()
    plt.clf()